In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install ../databricks_helpers
dbutils.library.restartPython()

In [0]:
import os
from databricks_helpers.databricks_helper import DatabricksHelpers
dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

silver_table_name = "moyalanjintos_catalog.aviation_analytics.silver_flight_weather"
gold_table_name = "moyalanjintos_catalog.aviation_analytics.old_airline_resiliance"


### Business Inferences
- Which airlines and airports are most impacted by winter weather, and who is the most resilient?
- Average Delay under high winds (>40kmph)
- Rain driven cancellations
- On-Time Rate (<15 min delay)

In [0]:
from pyspark.sql.functions import col, sum, avg, count, round, when

# 1. Read from the pristine Silver layer
silver_df = spark.read.table(silver_table_name)


# 2. Build the Inferences (Aggregating by Airport and Airline)
gold_weather_impact_df = (silver_df
    .groupBy("Origin", "Carrier")
    .agg(
        # Flight Metrics
        count("Flight_Number").alias("Total_Scheduled_Flights"),
        sum("Is_Cancelled").alias("Total_Cancellations"),
        round(avg("Departure_Delay"), 2).alias("Avg_Delay_Mins"),

        
        # Weather Inferences (Was there snow? How cold was it?)
        round(avg("Snow_Depth_cm"), 2).alias("Snowfall_Depth_cm"),
        round(avg("Avg_Temp_C"), 2).alias("Avg_Temperature_C"),
        round(avg("Total_Precipitation_mm"), 2).alias("Avg_Precipitation_mm"),
        round(avg("Avg_Wind_Speed_kmph"), 2).alias("Avg_Wind_Speed_kmh"),
        
        # On-Time Rate (Industry standard: < 15 mins delay is considered on-time)
        round(
            sum(when((col("Departure_Delay") < 15) & (col("Is_Cancelled") == 0), 1).otherwise(0)) 
            / count("Flight_Number") * 100, 2
        ).alias("On_Time_Performance_Pct"),

        # Complex Inference 1: Cancellation Rate during freezing temps (below 0C)
        round(
            sum(when((col("Avg_Temp_C") < 0) & (col("Is_Cancelled") == 1), 1).otherwise(0)) 
            / count("Flight_Number") * 100, 2
        ).alias("Freezing_Temp_Cancellation_Rate_Pct"),
        
        # Complex Inference 2: Delay Rate during High Winds (> 40 km/h)
        round(
            sum(when((col("Avg_Wind_Speed_kmph") > 40) & (col("Departure_Delay") >= 15), 1).otherwise(0)) 
            / sum(when(col("Avg_Wind_Speed_kmph") > 40, 1).otherwise(0.0001)) * 100, 2 # 0.0001 prevents divide-by-zero
        ).alias("High_Wind_Delay_Rate_Pct"),
        
        # Complex Inference 3: Cancellation Rate during Heavy Rain (> 10 mm)
        round(
            sum(when((col("Total_Precipitation_mm") > 10) & (col("Is_Cancelled") == 1), 1).otherwise(0)) 
            / sum(when(col("Total_Precipitation_mm") > 10, 1).otherwise(0.0001)) * 100, 2
        ).alias("Heavy_Rain_Cancellation_Rate_Pct")
    )
    # Filter out noise (e.g., airlines that only flew 2 times from an airport)
    .filter(col("Total_Scheduled_Flights") > 10) 
)



(gold_weather_impact_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table_name)
)


###Optimization
When you have a table that BI tools query heavily, the physical layout of the Parquet files matters immensely.

If a dashboard user filters for Origin = 'ORD', Spark usually has to scan every single file to find ORD flights. We fix this using Z-Ordering. Z-Ordering physically sorts the data inside the Parquet files so related information is colocated, allowing Spark to skip reading files that don't contain what it needs (Data Skipping).

In [0]:
spark.sql(f"""
    OPTIMIZE {gold_table_name} 
    ZORDER BY (Origin, Carrier)
""")

In [0]:
%sql
--Airline reliability dashboard, Who is the most reliable airline overall?
SELECT 
    Carrier,
    SUM(Total_Scheduled_Flights) AS Total_Flights,
    AVG(On_Time_Performance_Pct) AS Avg_On_Time_Pct
FROM moyalanjintos_catalog.aviation_analytics.gold_airline_resiliance
GROUP BY Carrier
ORDER BY Avg_On_Time_Pct DESC

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--Wind impact analysis, Does high wind actually cause delays, and how does it vary by airport?
SELECT 
    Origin AS Airport,
    AVG(Avg_Wind_Speed_kmh) AS Avg_Max_Wind_Speed,
    AVG(High_Wind_Delay_Rate_Pct) AS High_Wind_Delay_Rate
FROM moyalanjintos_catalog.aviation_analytics.gold_airline_resiliance
GROUP BY Origin
ORDER BY Avg_Max_Wind_Speed DESC

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--Freezing temperature effect
SELECT 
    Origin AS Airport,
    AVG(Freezing_Temp_Cancellation_Rate_Pct) AS Winter_Cancellation_Rate
FROM moyalanjintos_catalog.aviation_analytics.gold_airline_resiliance
WHERE Freezing_Temp_Cancellation_Rate_Pct > 0
GROUP BY Origin
ORDER BY Winter_Cancellation_Rate DESC

In [0]:
%sql
--Heavy rain cancellation effect, Which airports shut down operations fastest when the heavy rain starts?
SELECT 
    Origin AS Airport,
    AVG(Heavy_Rain_Cancellation_Rate_Pct) AS Rain_Cancellation_Rate
FROM moyalanjintos_catalog.aviation_analytics.gold_airline_resiliance
WHERE Heavy_Rain_Cancellation_Rate_Pct > 0
GROUP BY Origin
ORDER BY Rain_Cancellation_Rate DESC

Databricks visualization. Run in Databricks to view.